# Churn & Retention — EDA Cleaning Notebook


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv("../data/raw_churn_retention.csv")
df.head()


In [ ]:
df.shape
df.info()
df.describe()
df.isnull().sum()
df.duplicated().sum()


In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False).str.replace("-", "_", regex=False)


In [ ]:
for col in ["signup_date", "last_active_date", "churn_date"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

for col in ["tenure_days", "sessions_last_30d", "avg_session_duration", "feature_usage_score", "engagement_score", "revenue", "lifetime_value", "churn"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())


In [ ]:
for col in ["segment", "plan_type", "region", "device", "acquisition_channel", "churn_reason"]:
    df[col] = df[col].fillna("Unknown")

df = df.drop_duplicates()


In [ ]:
df["churn_flag"] = df["churn"].astype(int)
df["retention_flag"] = 1 - df["churn_flag"]
df["churn_rate"] = df["churn_flag"] * 100
df["retention_rate"] = df["retention_flag"] * 100
df["revenue_at_risk"] = df["revenue"] * df["churn_flag"]
df["engagement_band"] = pd.cut(df["engagement_score"], bins=[-1,25,50,100], labels=["Low Engagement", "Medium Engagement", "High Engagement"])
df["tenure_band"] = pd.cut(df["tenure_days"], bins=[-1,30,90,180,np.inf], labels=["0–30 Days", "31–90 Days", "91–180 Days", "181+ Days"])


In [ ]:
df.to_csv("../data/churn_retention.csv", index=False)
